In [ ]:
# %%
from __future__ import print_function

import numpy as np 
import matplotlib.pyplot as plt
import openpathsampling as paths

# we use the %run magic because this isn't in a package
%run ../Toy_potentials.py
%run ../../Resources/toy_plot_helpers.py

In [ ]:
import openpathsampling.engines.toy as toys
pes = potential_1_linear()
n_harmonics = pes.n_harmonics


In [ ]:
x = np.linspace(pes.extent[0],pes.extent[1],300)
y = np.linspace(pes.extent[2],pes.extent[3],300)
x_2d, y_2d, U = pes.plot_pes(x,y)
# Step 2: Create the colormap
colormap = 'plasma'  # You can choose a different colormap from Matplotlib's colormaps

# Step 3: Generate the 2D image

plt.figure()
plt.contour(x_2d,y_2d,U, levels = pes.levels)
plt.colorbar(label='Z-axis value')  # Add a colorbar to show the intensity scale
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.show()

In [ ]:

integ = toys.LangevinBAOABIntegrator(dt=0.02, temperature=0.1, gamma=2.5)
options={
    'integ' : integ,
    'n_frames_max' : 5000,
    'n_steps_per_frame' : 1
}
toy_eng = toys.Engine(
    options=options,
    topology=pes.topology
)
toy_eng.initialized = True

template = pes.template(toy_eng)
toy_eng.current_snapshot = template
paths.PathMover.engine = toy_eng

# Collective variables to define the states
opA = paths.CoordinateFunctionCV(name="opA", f=pes.stable_interface_function, center=pes.state_A)
opB = paths.CoordinateFunctionCV(name="opB", f=pes.stable_interface_function, center=pes.state_B)

# State volumes in CV space
stateA = paths.CVDefinedVolume(opA, 0.0, pes.state_boundary).named('StateA')
stateB = paths.CVDefinedVolume(opB, 0.0, pes.state_boundary).named('StateB')
#

In [ ]:
# Fake an initial trajectory
init_AB = paths.Trajectory(pes.simple_initial_path(100,toy_eng))


In [ ]:
def xval(snapshot):
    return snapshot.xyz[0][0]

cvX = paths.CoordinateFunctionCV(name="cvX", f=xval).with_diskcache()

In [ ]:
interface_value = -0.5

In [ ]:
print("Creating interfaces...")
interface = paths.VolumeInterfaceSet(cvX, minvals=-float("inf"), maxvals=interface_value) #for interface of paths going from A to B

In [ ]:
# %%
TIS = True
if TIS:
    network = paths.MISTISNetwork([(stateA, interface, stateB)]).named('mstis')
else:
    network = paths.TPSNetwork(stateA, stateB).named('tps')
move_scheme = paths.OneWayShootingMoveScheme(network, 
                                            engine=toy_eng).named("scheme")
# %%
initial_conditions = move_scheme.initial_conditions_from_trajectories(init_AB)

# %%
initial_conditions.sanity_check()


In [ ]:
# %%
storage = paths.Storage("simple_store.nc", "w", template=template)
storage.save(initial_conditions)
storage.save(toy_eng)
storage.save(network)
storage.save(move_scheme)


In [ ]:
sampler = paths.PathSampling(storage, move_scheme, initial_conditions)

n_run = 50
sampler.run(n_run)

In [ ]:
trajectories = [step.active[0].trajectory for step in storage.steps[:]]
plt.figure()
plt.contour(x_2d,y_2d,U, pes.levels)
plt.colorbar(label='Z-axis value') 
repcolordict = {0 : 'k-', 1 : 'r-', 2 : 'g-', 3 : 'b-', 4 : 'r-'}
for i, traj in enumerate(trajectories):
    plt.plot(traj.xyz[:,0,0], traj.xyz[:,0,1], repcolordict[i % 5])

states_plot = np.vectorize(CallableVolume(stateA))(x_2d, -y_2d)
states_plot += np.vectorize(CallableVolume(stateB))(x_2d, -y_2d)
plt.imshow(states_plot, extent=pes.extent, cmap="Blues",
            interpolation='nearest', vmin=0.0, vmax=2.0,
            aspect='auto')
 # Add a colorbar to show the intensity scale
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.show()

In [ ]:
storage.close()